# Advanced Pitch Statistics from Statcast

This notebook explores the advanced pitch-level data used to build pitcher and batter profiles.

**Pitcher Arsenal Stats:**
- Velocity, spin, movement by pitch type
- Pitch usage percentages
- Whiff%, CSW%, zone%, chase induced

**Batter Profile Stats:**
- Performance vs pitch types (fastball, breaking, offspeed)
- Performance vs velocity buckets (90-93, 94-96, 97+)
- Performance vs movement profiles (high rise, heavy sweep, heavy drop)
- Contact quality (exit velo, xwOBA)

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np

from mlb_data import (
    get_statcast_pitches,
    get_pitcher_arsenal,
    get_pitcher_pitch_type_stats,
    get_batter_pitch_stats,
    get_batter_pitch_type_stats,
    get_plate_appearances,
)

# Import config
from src.config import (
    DATA_START,
    DATA_END,
    DATA_DIR,
)

pd.set_option('display.max_columns', 50)

print(f"Date range: {DATA_START} to {DATA_END}")

Date range: 2021-04-01 to 2026-04-09


In [ ]:
# Memory management utilities
import gc
import psutil

def clear_mem():
    """Run garbage collection and print memory usage."""
    gc.collect()
    mem_gb = psutil.Process().memory_info().rss / 1024**3
    print(f"Memory: {mem_gb:.2f} GB")
    return mem_gb

clear_mem()

## 1. Raw Statcast Data

First, let's fetch the raw pitch-level data and explore what's available.

In [2]:
from pathlib import Path

# Load pitches from notebook 01 output
pitches_path = Path(f"../{DATA_DIR}/raw/pitches.parquet")

if pitches_path.exists():
    pitches = pd.read_parquet(pitches_path)
    print(f"Loaded pitches from {pitches_path}")
else:
    print(f"Pitches file not found at {pitches_path}")
    print("Run notebook 01_compile_data first to fetch the data.")
    raise FileNotFoundError(f"Run notebook 01 first: {pitches_path}")

print(f"Shape: {pitches.shape}")
print(f"Date range: {pitches['game_date'].min()} to {pitches['game_date'].max()}")
pitches.head()

clear_mem()

Loaded pitches from ../data/raw/pitches.parquet
Shape: (3896176, 118)
Date range: 2021-04-01 00:00:00 to 2026-04-08 00:00:00


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,...,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,ST,2026-04-08,83.5,-1.2,5.34,"Watson, Ryan",663604,670245,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,5,Brandon Lockridge flies out to center fielder ...,R,R,R,BOS,MIL,X,8,fly_ball,0,...,0.217,96.6,5,-5,0.999,0.001,28,29,29,29,1,3,<NA>,<NA>,<NA>,<NA>,2.46,-1.04,-1.04,<NA>,11.670274,-2.874085,28.222737,41.159652,35.898104
1,SI,2026-04-08,92.5,-0.94,5.43,"Watson, Ryan",663604,670245,None,called_strike,<NA>,<NA>,<NA>,<NA>,3,Called Strike,R,R,R,BOS,MIL,S,<NA>,None,0,...,0.043,<NA>,5,-5,0.999,0.001,28,29,29,29,1,3,<NA>,<NA>,<NA>,<NA>,2.11,1.29,1.29,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,FF,2026-04-08,93.9,-1.1,5.46,"Watson, Ryan",650859,670245,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,8,"Luis Rengifo grounds out, second baseman Isiah...",R,L,R,BOS,MIL,X,4,ground_ball,3,...,0.332,95.2,5,-5,0.997,0.003,28,29,29,29,1,3,<NA>,<NA>,<NA>,<NA>,1.33,0.54,-0.54,<NA>,11.358738,0.220119,34.835735,32.976146,32.020468
3,CU,2026-04-08,81.8,-1.14,5.37,"Watson, Ryan",650859,670245,None,ball,<NA>,<NA>,<NA>,<NA>,13,Ball,R,L,R,BOS,MIL,B,<NA>,None,2,...,-0.104,<NA>,5,-5,0.998,0.002,28,29,29,29,1,3,<NA>,<NA>,<NA>,<NA>,4.69,-0.93,0.93,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,FF,2026-04-08,94.1,-1.22,5.43,"Watson, Ryan",650859,670245,None,foul,<NA>,<NA>,<NA>,<NA>,12,Foul,R,L,R,BOS,MIL,S,<NA>,None,2,...,0.0,88.0,5,-5,0.998,0.002,28,29,29,29,1,3,<NA>,<NA>,<NA>,<NA>,1.22,0.53,-0.53,<NA>,-1.577897,19.311881,20.563809,33.144856,19.846114


In [3]:
# Key pitch characteristic columns
pitch_cols = [
    'pitch_type', 'release_speed', 'release_spin_rate',
    'pfx_x', 'pfx_z', 'release_extension',
    'plate_x', 'plate_z', 'zone',
    'description', 'events', 'type',
]

pitches[pitch_cols].head(20)

# Free memory - raw pitches no longer needed after exploration
del pitches
clear_mem()

,pitch_type,release_speed,release_spin_rate,pfx_x,pfx_z,release_extension,plate_x,plate_z,zone,description,events,type
0,ST,83.5,2104,1.04,0.84,7.0,0.145121,2.456601,5,hit_into_play,field_out,X
1,SI,92.5,2134,-1.29,0.54,7.1,0.61734,2.758271,3,called_strike,None,S
2,FF,93.9,2246,-0.54,1.28,7.1,-0.012027,1.677773,8,hit_into_play,field_out,X
3,CU,81.8,2083,0.93,-1.22,6.9,-0.87773,0.638529,13,ball,None,B
4,FF,94.1,2181,-0.53,1.4,7.1,0.022882,3.366338,12,foul,None,S
5,CU,83.1,2221,1.1,-1.09,6.8,-0.171998,0.378888,13,ball,None,B
6,CU,82.5,2242,1.25,-0.97,7.1,0.194652,0.81377,14,ball,None,B
7,CU,82.0,2256,1.37,-1.1,6.9,0.086453,0.57079,14,swinging_strike,None,S
8,CU,81.3,2182,1.2,-1.04,7.1,-0.174587,2.012696,8,called_strike,None,S
9,CU,80.4,2104,1.31,-1.02,6.9,-0.381306,2.268665,4,hit_into_play,field_out,X


In [4]:
# Pitch type distribution
print("Pitch type distribution:")
pitch_counts = pitches['pitch_type'].value_counts()
pitch_pcts = pitches['pitch_type'].value_counts(normalize=True) * 100

pd.DataFrame({
    'count': pitch_counts,
    'pct': pitch_pcts.round(1)
}).head(15)

Pitch type distribution:


,count,pct
pitch_type,,
FF,1254195,32.9
SL,599345,15.7
SI,592157,15.5
CH,409228,10.7
FC,293496,7.7
CU,260746,6.8
ST,206746,5.4
FS,90605,2.4
KC,76740,2.0


In [ ]:
# Pitch outcome descriptions
print("Pitch descriptions (outcomes):")
pitches['description'].value_counts()

Pitch descriptions (outcomes):


description
ball                       1293728
foul                        687897
hit_into_play               683364
called_strike               634082
swinging_strike             418606
blocked_ball                 84767
foul_tip                     38373
swinging_strike_blocked      23184
automatic_ball               12009
hit_by_pitch                 11389
foul_bunt                     6531
missed_bunt                   1242
automatic_strike               628
pitchout                       237
bunt_foul_tip                  136
foul_pitchout                    2
intent_ball                      1
Name: count, dtype: int64

: 

## 2. Pitcher Arsenal Profiles

Aggregate pitch-level data into pitcher profiles with:
- Fastball characteristics (velocity, spin, movement)
- Breaking ball characteristics
- Offspeed characteristics
- Overall effectiveness metrics (whiff%, CSW%, etc.)

In [ ]:
# Get pitcher arsenal stats (uses cached pitches)
pitcher_arsenal = get_pitcher_arsenal(
    DATA_START, DATA_END,
    min_pitches=50,  # Higher threshold for multi-season data
    pitches_df=pitches
)

print(f"Pitchers with arsenal data: {len(pitcher_arsenal)}")
pitcher_arsenal.head(10)

clear_mem()

Computing pitcher arsenal stats...


In [ ]:
# All available arsenal columns
print("Pitcher arsenal columns:")
print(pitcher_arsenal.columns.tolist())

Pitcher arsenal columns:
['pitcher_id', 'pitcher_name', 'p_throws', 'total_pitches', 'primary_pitch', 'ff_usage_pct', 'si_usage_pct', 'si_velo_avg', 'si_spin_avg', 'si_vmov_avg', 'si_hmov_avg', 'si_whiff_pct', 'fc_usage_pct', 'sl_usage_pct', 'sl_velo_avg', 'sl_spin_avg', 'sl_vmov_avg', 'sl_hmov_avg', 'sl_whiff_pct', 'cu_usage_pct', 'ch_usage_pct', 'sv_usage_pct', 'fs_usage_pct', 'kc_usage_pct', 'st_usage_pct', 'kn_usage_pct', 'whiff_pct', 'swstr_pct', 'csw_pct', 'zone_pct', 'chase_pct_induced', 'contact_pct', 'f_strike_pct', 'ff_velo_avg', 'ff_spin_avg', 'ff_vmov_avg', 'ff_hmov_avg', 'ff_whiff_pct', 'fc_velo_avg', 'fc_spin_avg', 'fc_vmov_avg', 'fc_hmov_avg', 'fc_whiff_pct', 'cu_velo_avg', 'cu_spin_avg', 'cu_vmov_avg', 'cu_hmov_avg', 'cu_whiff_pct', 'ch_velo_avg', 'ch_spin_avg', 'ch_vmov_avg', 'ch_hmov_avg', 'ch_whiff_pct', 'st_velo_avg', 'st_spin_avg', 'st_vmov_avg', 'st_hmov_avg', 'st_whiff_pct', 'fs_velo_avg', 'fs_spin_avg', 'fs_vmov_avg', 'fs_hmov_avg', 'fs_whiff_pct', 'kc_velo_avg'

In [ ]:
# Summary statistics for key arsenal metrics
arsenal_stats = [
    'fb_velo_avg', 'fb_velo_max', 'fb_spin_avg', 'fb_vmov_avg',
    'brk_velo_avg', 'brk_hmov_avg', 'brk_vmov_avg',
    'off_velo_avg', 'off_velo_diff',
    'fb_usage_pct', 'brk_usage_pct', 'off_usage_pct',
    'whiff_pct', 'csw_pct', 'zone_pct', 'chase_pct_induced',
]

available_stats = [c for c in arsenal_stats if c in pitcher_arsenal.columns]
pitcher_arsenal[available_stats].describe().round(2)

,whiff_pct,csw_pct,zone_pct,chase_pct_induced
count,1622.00,1622.00,1622.00,1622.00
mean,0.23,0.27,0.48,0.27
std,0.06,0.04,0.04,0.05
min,0.00,0.06,0.17,0.03
25%,0.19,0.25,0.46,0.24
50%,0.23,0.27,0.49,0.27
75%,0.26,0.29,0.51,0.30
max,0.53,0.38,0.68,0.45


: 

## 3. Pitcher Stats by Pitch Type

Detailed breakdown for each pitch type a pitcher throws.

In [ ]:
# Get per-pitch-type stats for pitchers
pitcher_pitch_types = get_pitcher_pitch_type_stats(
    DATA_START, DATA_END,
    min_pitches=50,  # Higher threshold for multi-season data
    pitches_df=pitches
)

print(f"Pitcher-pitch type combinations: {len(pitcher_pitch_types)}")
pitcher_pitch_types.head(15)

Computing pitcher pitch-type stats...


In [ ]:
# Best sliders by whiff rate
sliders = pitcher_pitch_types[pitcher_pitch_types['pitch_type'] == 'SL']
print("Top 10 sliders by whiff rate:")
sliders.nlargest(10, 'whiff_pct')[[
    'pitcher_name', 'pitch_count', 'usage_pct',
    'velo_avg', 'hmov_avg', 'vmov_avg', 'whiff_pct'
]]

Top 10 sliders by whiff rate:


,pitcher_name,pitch_count,usage_pct,velo_avg,hmov_avg,vmov_avg,whiff_pct
4428,"Garcia, Luis",102,0.017152,85.039216,0.458725,0.036569,0.633333
463,"Ramírez, Erasmo",55,0.017805,81.480000,0.309455,0.391091,0.612903
429,"Hendriks, Liam",655,0.261581,88.499695,0.196718,0.295802,0.591900
4408,"Crochet, Garrett",50,0.006972,86.346000,-0.612400,0.352800,0.588235
3375,"Felipe, Angel",96,0.354244,83.371875,0.258646,-0.388958,0.575758
4498,"Ferrer, Jose A.",147,0.062901,88.721769,-0.099388,0.176122,0.561404
1923,"Hader, Josh",1503,0.290659,84.393147,-0.240299,0.109894,0.557769
2256,"Williams, Devin",53,0.011160,85.264151,0.444906,0.472264,0.545455
714,"Brothers, Rex",344,0.344689,86.179942,-0.085174,0.114302,0.533333
1679,"Reyes, Alex",353,0.281724,86.404816,0.688612,-0.023541,0.532051


In [ ]:
# Best changeups by whiff rate
changeups = pitcher_pitch_types[pitcher_pitch_types['pitch_type'] == 'CH']
print("Top 10 changeups by whiff rate:")
changeups.nlargest(10, 'whiff_pct')[[
    'pitcher_name', 'pitch_count', 'usage_pct',
    'velo_avg', 'vmov_avg', 'whiff_pct'
]]

Top 10 changeups by whiff rate:


,pitcher_name,pitch_count,usage_pct,velo_avg,vmov_avg,whiff_pct
4220,"Criswell, Jeff",51,0.144068,86.778431,0.466471,0.550000
2157,"Kinley, Tyler",106,0.024357,87.357547,0.923302,0.540541
1675,"Reyes, Alex",114,0.090982,90.335965,0.823684,0.537037
3990,"Morejon, Adrian",95,0.030596,90.412632,0.321158,0.518519
5318,"Miller, Mason",86,0.031583,91.960465,0.503837,0.517241
5360,"Rodriguez, Bradgley",60,0.284360,88.725000,0.806500,0.515152
1680,"Ferguson, Tyler",61,0.033134,86.945902,0.156557,0.500000
2312,"Fernández, Julian",55,0.246637,87.089091,0.633273,0.500000
4464,"Perkins, Jack",74,0.104965,89.429730,0.038243,0.500000
5200,"Pérez, Eury",325,0.093017,89.725538,0.784031,0.500000


## 4. Batter Pitch Performance Profiles

How batters perform against different pitch characteristics:
- By pitch type group (fastball, breaking, offspeed)
- By velocity bucket (90-93, 94-96, 97+)
- By movement type (high rise, heavy sweep, heavy drop)
- By pitcher handedness (vs LHP, vs RHP)

In [ ]:
# Get batter pitch stats (uses cached pitches)
batter_profiles = get_batter_pitch_stats(
    DATA_START, DATA_END,
    min_pitches=50,  # Higher threshold for multi-season data
    pitches_df=pitches
)

print(f"Batters with pitch profiles: {len(batter_profiles)}")
batter_profiles.head(10)

clear_mem()

Computing batter pitch stats...
  Computed pitch stats for 1291 batters
Batters with pitch profiles: 1291


,batter_id,batter_name,stand,total_pitches,swing_pct,whiff_pct,contact_pct,csw_pct,zone_swing_pct,chase_pct,avg_exit_velo,max_exit_velo,avg_launch_angle,xwoba,xba,barrel_pct,vs_ff_whiff_pct,vs_ff_contact_pct,vs_ff_xwoba,vs_si_whiff_pct,vs_si_contact_pct,vs_si_xwoba,vs_fc_whiff_pct,vs_fc_contact_pct,vs_fc_xwoba,...,vs_st_xwoba,velo_90_93_pitches,velo_90_93_whiff_pct,velo_90_93_swing_pct,velo_94_96_pitches,velo_94_96_whiff_pct,velo_94_96_swing_pct,velo_97_plus_pitches,velo_97_plus_whiff_pct,velo_97_plus_swing_pct,vs_LHP_pitches,vs_LHP_whiff_pct,vs_LHP_xwoba,vs_RHP_pitches,vs_RHP_whiff_pct,vs_RHP_xwoba,vs_fs_whiff_pct,vs_fs_contact_pct,vs_fs_xwoba,vs_sv_whiff_pct,vs_sv_contact_pct,vs_sv_xwoba,vs_kn_whiff_pct,vs_kn_contact_pct,vs_kn_xwoba
0,405395,,R,2620,0.478244,0.198723,0.801277,0.269466,0.635703,0.337192,90.773828,112.9,13.892578,0.381008,0.328104,0.033138,0.115741,0.884259,0.415883,0.129353,0.870647,0.35543,0.123596,0.876404,0.350156,...,0.183397,645.0,0.109375,0.496124,403.0,0.101770,0.560794,131.0,0.222222,0.618321,1151.0,0.182143,0.459617,1469.0,0.212121,0.318509,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,408234,,R,5215,0.493576,0.238539,0.761461,0.272291,0.697802,0.298463,90.282476,112.6,7.739769,0.347429,0.328076,0.023958,0.197015,0.802985,0.388774,0.103529,0.896471,0.277016,0.309278,0.690722,0.276434,...,0.331420,1569.0,0.185829,0.476737,922.0,0.151822,0.535792,279.0,0.226519,0.648746,1471.0,0.235469,0.355101,3744.0,0.239622,0.344737,0.454545,0.545455,0.252891,NaN,NaN,NaN,NaN,NaN,NaN
2,425784,,R,286,0.503497,0.368056,0.631944,0.360140,0.656051,0.317829,84.883721,106.5,8.581395,0.392586,0.337121,0.000000,0.280000,0.720000,0.499254,0.333333,0.666667,0.2765,NaN,NaN,NaN,...,NaN,91.0,0.339623,0.582418,46.0,0.347826,0.500000,NaN,NaN,NaN,125.0,0.396825,0.461009,161.0,0.345679,0.342012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,425794,,R,287,0.567944,0.306748,0.693252,0.331010,0.701389,0.433566,63.797959,109.8,-15.551020,0.253683,0.235565,0.020408,0.188406,0.811594,0.360073,0.240000,0.760000,0.18019,NaN,NaN,NaN,...,NaN,83.0,0.173077,0.626506,48.0,0.259259,0.562500,NaN,NaN,NaN,68.0,0.315789,0.147900,219.0,0.304000,0.29776,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,425877,,R,2711,0.576909,0.213555,0.786445,0.249723,0.758136,0.396615,86.694343,108.2,15.911814,0.323888,0.307536,0.016611,0.210325,0.789675,0.368055,0.098859,0.901141,0.274684,0.310606,0.689394,0.277276,...,0.368370,677.0,0.189526,0.592319,510.0,0.203175,0.617647,128.0,0.253012,0.648438,528.0,0.205224,0.343592,2183.0,0.215278,0.319075,0.315789,0.684211,0.178000,NaN,NaN,NaN,NaN,NaN,NaN
5,429664,,L,465,0.565591,0.201521,0.798479,0.234409,0.743119,0.408907,88.617895,112.0,4.484211,0.289116,0.303088,0.010526,0.184783,0.815217,0.290449,0.027778,0.972222,0.262162,0.071429,0.928571,0.208556,...,NaN,122.0,0.135135,0.606557,77.0,0.210526,0.493506,30.0,0.125000,0.533333,NaN,NaN,NaN,423.0,0.200837,0.291319,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,435559,,R,1529,0.485284,0.192722,0.807278,0.251799,0.692206,0.282383,84.805051,105.0,17.875421,0.273133,0.256906,0.013378,0.105263,0.894737,0.266977,0.061538,0.938462,0.287879,0.108108,0.891892,0.310282,...,0.030400,402.0,0.091837,0.487562,276.0,0.103448,0.525362,75.0,0.100000,0.533333,552.0,0.194553,0.281681,977.0,0.191753,0.267803,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,443558,,R,4877,0.511995,0.293152,0.706848,0.279680,0.724481,0.320468,91.758412,117.5,9.856805,0.411106,0.343142,0.023585,0.218942,0.781058,0.455874,0.156915,0.843085,0.369275,0.248731,0.751269,0.424703,...,0.312097,1188.0,0.215488,0.500000,901.0,0.207627,0.523862,273.0,0.246575,0.534799,1801.0,0.279476,0.431768,3076.0,0.301075,0.398756,0.407407,0.592593,0.304596,NaN,NaN,NaN,NaN,NaN,NaN
8,444482,,L,6682,0.500299,0.220461,0.779539,0.263095,0.697071,0.310811,89.499218,113.8,9.453479,0.351275,0.323065,0.015601,0.155689,0.844311,0.377505,0.088670,0.911330,0.343568,0.170213,0.829787,0.383771,...,0.337606,1619.0,0.162634,0.459543,1379.0,0.143243,0.536621,355.0,0.153465,0.569014,924.0,0.230769,0.3

In [ ]:
# All available batter profile columns
print("Batter profile columns:")
print(batter_profiles.columns.tolist())

Batter profile columns:
['batter_id', 'batter_name', 'stand', 'total_pitches', 'swing_pct', 'whiff_pct', 'contact_pct', 'csw_pct', 'zone_swing_pct', 'chase_pct', 'avg_exit_velo', 'max_exit_velo', 'avg_launch_angle', 'xwoba', 'xba', 'barrel_pct', 'vs_ff_whiff_pct', 'vs_ff_contact_pct', 'vs_ff_xwoba', 'vs_si_whiff_pct', 'vs_si_contact_pct', 'vs_si_xwoba', 'vs_fc_whiff_pct', 'vs_fc_contact_pct', 'vs_fc_xwoba', 'vs_sl_whiff_pct', 'vs_sl_contact_pct', 'vs_sl_xwoba', 'vs_cu_whiff_pct', 'vs_cu_contact_pct', 'vs_cu_xwoba', 'vs_ch_whiff_pct', 'vs_ch_contact_pct', 'vs_ch_xwoba', 'vs_kc_whiff_pct', 'vs_kc_contact_pct', 'vs_kc_xwoba', 'vs_st_whiff_pct', 'vs_st_contact_pct', 'vs_st_xwoba', 'velo_90_93_pitches', 'velo_90_93_whiff_pct', 'velo_90_93_swing_pct', 'velo_94_96_pitches', 'velo_94_96_whiff_pct', 'velo_94_96_swing_pct', 'velo_97_plus_pitches', 'velo_97_plus_whiff_pct', 'velo_97_plus_swing_pct', 'vs_LHP_pitches', 'vs_LHP_whiff_pct', 'vs_LHP_xwoba', 'vs_RHP_pitches', 'vs_RHP_whiff_pct', 'vs_RH

In [ ]:
# Summary of key batter metrics
batter_stats = [
    'whiff_pct', 'contact_pct', 'chase_pct', 'zone_swing_pct',
    'fastball_whiff_pct', 'breaking_whiff_pct', 'offspeed_whiff_pct',
    'velo_97_plus_whiff_pct', 'high_rise_whiff_pct', 'heavy_sweep_whiff_pct',
    'avg_exit_velo', 'xwoba',
]

available_stats = [c for c in batter_stats if c in batter_profiles.columns]
batter_profiles[available_stats].describe().round(3)

,whiff_pct,contact_pct,chase_pct,zone_swing_pct,velo_97_plus_whiff_pct,avg_exit_velo,xwoba
count,1291.000,1291.000,1291.000,1291.000,839.000,1079.000,1079.000
mean,0.260,0.740,0.292,0.670,0.221,86.733,0.350
std,0.076,0.076,0.071,0.070,0.084,4.592,0.060
min,0.063,0.392,0.070,0.351,0.000,55.809,0.128
25%,0.211,0.697,0.245,0.632,0.163,85.356,0.312
50%,0.253,0.747,0.289,0.673,0.215,87.537,0.348
75%,0.303,0.789,0.331,0.716,0.270,89.339,0.388
max,0.608,0.937,0.681,0.900,0.562,95.963,0.623


In [ ]:
# Best contact rates (lowest whiff)
print("Top 10 contact rates (lowest whiff%):")
batter_profiles.nsmallest(10, 'whiff_pct')[[
    'batter_id', 'stand', 'total_pitches',
    'whiff_pct', 'contact_pct', 'chase_pct', 'avg_exit_velo'
]]

Top 10 contact rates (lowest whiff%):


,batter_id,stand,total_pitches,whiff_pct,contact_pct,chase_pct,avg_exit_velo
550,650333,L,11941,0.063486,0.936514,0.297756,86.945622
214,591971,L,411,0.065359,0.934641,0.179104,84.010390
306,605200,R,153,0.066667,0.933333,0.115385,65.558333
1037,680757,L,11175,0.080725,0.919275,0.207019,84.903060
656,663611,R,3093,0.085106,0.914894,0.294194,84.802029
11,445055,L,54,0.085714,0.914286,0.518519,NaN
1263,800325,R,87,0.093023,0.906977,0.238095,NaN
1277,805779,R,2209,0.094033,0.905967,0.332726,84.797683
691,664058,R,3335,0.096263,0.903737,0.303915,81.300231
741,665877,R,977,0.096447,0.903553,0.229614,84.552551


In [ ]:
# Batters who struggle vs 97+ velocity
if 'velo_97_plus_whiff_pct' in batter_profiles.columns:
    has_velo_data = batter_profiles['velo_97_plus_whiff_pct'].notna()
    print("Batters who struggle most vs 97+ mph:")
    batter_profiles[has_velo_data].nlargest(10, 'velo_97_plus_whiff_pct')[[
        'batter_id', 'stand', 'velo_97_plus_pitches',
        'velo_97_plus_whiff_pct', 'whiff_pct'
    ]]

Batters who struggle most vs 97+ mph:


In [ ]:
# Batters who struggle vs breaking balls
if 'breaking_whiff_pct' in batter_profiles.columns:
    has_brk_data = batter_profiles['breaking_whiff_pct'].notna()
    print("Batters who struggle most vs breaking balls:")
    batter_profiles[has_brk_data].nlargest(10, 'breaking_whiff_pct')[[
        'batter_id', 'stand', 'breaking_pitches',
        'breaking_whiff_pct', 'whiff_pct'
    ]]

## 5. Batter Stats by Pitch Type

Detailed breakdown for each pitch type a batter faces.

In [ ]:
# Get per-pitch-type stats for batters
batter_pitch_types = get_batter_pitch_type_stats(
    DATA_START, DATA_END,
    min_pitches=50,  # Higher threshold for multi-season data
    pitches_df=pitches
)

print(f"Batter-pitch type combinations: {len(batter_pitch_types)}")
batter_pitch_types.head(15)

Computing batter pitch-type stats...
  Computed stats for 7114 batter-pitch combinations
Batter-pitch type combinations: 7114


,batter_id,batter_name,stand,pitch_type,pitch_count,pct_of_pitches_seen,swing_pct,whiff_pct,contact_pct,avg_exit_velo,avg_launch_angle,xwoba,xba
0,405395,,R,CH,232,0.088550,0.461207,0.289720,0.710280,90.377778,4.361111,0.438562,0.391633
1,405395,,R,CU,231,0.088168,0.341991,0.354430,0.645570,85.900000,15.838710,0.322692,0.311740
2,405395,,R,FC,169,0.064504,0.526627,0.123596,0.876404,93.368421,12.210526,0.350156,0.308661
3,405395,,R,FF,777,0.296565,0.555985,0.115741,0.884259,92.598930,22.764706,0.415883,0.342693
4,405395,,R,KC,53,0.020229,0.377358,0.550000,0.450000,94.866667,4.666667,0.48062,0.459906
5,405395,,R,SI,456,0.174046,0.440789,0.129353,0.870647,92.335577,5.528846,0.35543,0.331774
6,405395,,R,SL,550,0.209924,0.463636,0.298039,0.701961,87.274444,6.544444,0.361531,0.302082
7,405395,,R,ST,108,0.041221,0.462963,0.260000,0.740000,82.161538,37.307692,0.183397,0.145713
8,408234,,R,CH,409,0.078428,0.486553,0.286432,0.713568,90.449367,4.468354,0.39906,0.377545
9,408234,,R,CU,252,0.048322,0.376984,0.305263,0.694737,87.730952,5.238095,0.331315,0.322911


In [ ]:
# Best fastball hitters (lowest whiff on FF)
ff_stats = batter_pitch_types[batter_pitch_types['pitch_type'] == 'FF']
if len(ff_stats) > 0 and 'xwoba' in ff_stats.columns:
    ff_qualified = ff_stats[ff_stats['pitch_count'] >= 100]
    print("Best fastball hitters by xwOBA:")
    ff_qualified.nlargest(10, 'xwoba')[[
        'batter_id', 'pitch_count', 'whiff_pct', 'contact_pct',
        'avg_exit_velo', 'xwoba'
    ]]

Best fastball hitters by xwOBA:


TypeError: Column 'xwoba' has dtype object, cannot use method 'nlargest' with this dtype

## 6. Plate Appearance Outcomes

Extract completed plate appearances with outcomes (K, BB, 1B, 2B, 3B, HR, OUT).

In [ ]:
# Get plate appearances with outcomes
plate_apps = get_plate_appearances(
    DATA_START, DATA_END,
    pitches_df=pitches
)

print(f"Total plate appearances: {len(plate_apps):,}")
plate_apps.head(15)

clear_mem()

Extracting 1,003,356 plate appearances...
Total plate appearances: 1,003,356


,game_pk,game_date,at_bat_number,inning,outs_when_up,pitcher_id,pitcher_name,p_throws,batter_id,stand,home_team,away_team,events,outcome,launch_speed,launch_angle,estimated_ba_using_speedangle,estimated_woba_using_speedangle,is_home_batter
0,634614,2021-04-14,77,9,2,445276,"Jansen, Kenley",R,606132,L,LAD,COL,strikeout,K,<NA>,<NA>,<NA>,0.0,True
4,634614,2021-04-14,76,9,1,445276,"Jansen, Kenley",R,641658,R,LAD,COL,strikeout,K,<NA>,<NA>,<NA>,0.0,True
11,634614,2021-04-14,75,9,0,445276,"Jansen, Kenley",R,676701,R,LAD,COL,strikeout,K,<NA>,<NA>,<NA>,0.0,True
14,634614,2021-04-14,74,9,0,445276,"Jansen, Kenley",R,624513,L,LAD,COL,walk,BB,<NA>,<NA>,<NA>,0.691717,True
19,634614,2021-04-14,73,8,2,453268,"Bard, Daniel",R,457759,R,LAD,COL,strikeout,K,<NA>,<NA>,<NA>,0.0,True
23,634614,2021-04-14,72,8,1,453268,"Bard, Daniel",R,608369,L,LAD,COL,field_out,OUT,81.2,-32,0.052,0.064,True
26,634614,2021-04-14,71,8,1,453268,"Bard, Daniel",R,605141,R,LAD,COL,walk,BB,<NA>,<NA>,<NA>,0.691717,True
32,634614,2021-04-14,70,8,1,453268,"Bard, Daniel",R,670042,L,LAD,COL,double,2B,90.2,19,0.521,0.495,True
34,634614,2021-04-14,69,8,0,453268,"Bard, Daniel",R,605131,R,LAD,COL,field_out,OUT,100.3,-13,0.203,0.184,True
35,634614,2021-04-14,68,8,0,453268,"Bard, Daniel",R,656716,L,LAD,COL,home_run,HR,101.7,29,0.628319,1.240638,True


In [ ]:
# Outcome distribution
print("PA outcome distribution:")
outcome_counts = plate_apps['outcome'].value_counts()
outcome_pcts = plate_apps['outcome'].value_counts(normalize=True) * 100

pd.DataFrame({
    'count': outcome_counts,
    'pct': outcome_pcts.round(1)
})

PA outcome distribution:


,count,pct
outcome,,
OUT,457772,45.6
K,227921,22.7
1B,141405,14.1
BB,81930,8.2
2B,43555,4.3
HR,30877,3.1
HBP,11292,1.1
OTHER,4895,0.5
3B,3709,0.4


In [ ]:
# Unique matchup counts
print(f"Unique pitchers: {plate_apps['pitcher_id'].nunique():,}")
print(f"Unique batters: {plate_apps['batter_id'].nunique():,}")

# Pitcher-batter combinations
matchups = plate_apps.groupby(['pitcher_id', 'batter_id']).size()
print(f"Unique pitcher-batter matchups: {len(matchups):,}")
print(f"\nPAs per matchup distribution:")
matchups.describe()

Unique pitchers: 2,802
Unique batters: 3,490
Unique pitcher-batter matchups: 364,350

PAs per matchup distribution:


count    364350.000000
mean          2.753825
std           2.919446
min           1.000000
25%           1.000000
50%           2.000000
75%           3.000000
max          53.000000
dtype: float64

## 7. Save Data

Save the computed profiles for use in model training.

In [ ]:
from pathlib import Path

# Create output directory
output_dir = Path(f'../{DATA_DIR}/profiles')
output_dir.mkdir(parents=True, exist_ok=True)

# Save profiles
pitcher_arsenal.to_csv(output_dir / 'pitcher_arsenal.csv', index=False)
pitcher_pitch_types.to_csv(output_dir / 'pitcher_pitch_types.csv', index=False)
batter_profiles.to_csv(output_dir / 'batter_profiles.csv', index=False)
batter_pitch_types.to_csv(output_dir / 'batter_pitch_types.csv', index=False)
plate_apps.to_parquet(output_dir / 'plate_appearances.parquet', index=False)

print(f"Saved data to {output_dir.resolve()}:")
print(f"  - pitcher_arsenal.csv: {len(pitcher_arsenal):,} rows")
print(f"  - pitcher_pitch_types.csv: {len(pitcher_pitch_types):,} rows")
print(f"  - batter_profiles.csv: {len(batter_profiles):,} rows")
print(f"  - batter_pitch_types.csv: {len(batter_pitch_types):,} rows")
print(f"  - plate_appearances.parquet: {len(plate_apps):,} rows")

clear_mem()

Saved data to /Users/matthewgillies/PitcherGamePreds/data/profiles:
  - pitcher_arsenal.csv: 1,618 rows
  - pitcher_pitch_types.csv: 5,516 rows
  - batter_profiles.csv: 1,291 rows
  - batter_pitch_types.csv: 7,114 rows
  - plate_appearances.parquet: 1,003,356 rows
